# IBM Stock Data Preprocessing

This notebook takes the raw IBM stock data and prepares it for the RNN/LSTM/GRU models. It runs through cleaning, feature engineering, splitting by year, scaling, and turning the data into sequences ready for training.

All the actual logic lives in `src/data_preprocessing.py`, this notebook just calls the functions and inspects the output.

In [ ]:
# path setup so we can import from src
import os
import sys
sys.path.append(os.path.abspath('..'))

import matplotlib.pyplot as plt
import seaborn as sns
from src.data_preprocessing import preprocess_pipeline, prepare_model_data
from src.utils import load_data

## Load the raw data

In [ ]:
df = load_data("../data/raw/ibm_stock_1980_2025.csv")
df

## Run the preprocessing pipeline

This single function handles everything in order:
- removes duplicate dates
- fixes the Volume column (string with commas to float)
- removes rows with negative or zero prices
- removes rows with invalid OHLC (High/Low logic errors)
- adds the engineered features (Daily Return, Volatility, High Low Range, Open Close Range)
- removes extreme return outliers
- saves the cleaned data to `data/processed/`

Each step prints what it found and removed, so if nothing needs cleaning it will say so instead of just running silently.

In [ ]:
df = preprocess_pipeline(df, '../data/processed')

## Quick look at the processed data

In [ ]:
df.head(10)

In [ ]:
df.tail(10)

## Split, scale, and create sequences

`prepare_model_data` does the full remaining pipeline:
- splits the data into train/valid/test by year
- scales everything with MinMaxScaler (fit only on train, to avoid leakage)
- builds the sliding window sequences (timesteps) for the RNN models
- reshapes into the (samples, timesteps, features) shape Keras expects
- saves all splits into one pickle file for reuse

Train covers up to 2022, valid is 2023 to 2024, test is 2025. Timesteps is set to 90 days per sequence.

In [ ]:
X_train, y_train, X_valid, y_valid, X_test, y_test = prepare_model_data(data=df,
                                                                        year_splits=[2022, 2024, 2025],
                                                                        timesteps=90, 
                                                                        num_features=len(df.columns), save_dir='../data/processed')

## Check the shapes
Makes sure the sequence shapes look right before moving to modeling.

In [ ]:
X_train.shape

In [ ]:
X_valid.shape

In [ ]:
X_test.shape

## Correlation after feature engineering
See how the new features (Daily Return, Volatility, ranges) relate to price and to each other.

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Matrix")
plt.show()

## Outliers check after cleaning
Boxplots on the final processed columns, just to confirm the cleaning worked as expected.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
cols_to_plot = df.columns

for ax, col in zip(axes.flatten(), cols_to_plot):
    sns.boxplot(y=df[col], ax=ax)
    ax.set_title(col)

plt.tight_layout()
plt.show()